# YOLOv8n -- Sofia domain-adaptation fine-tuning (notebook 11)

Domain-adaptation stage: starts from the RDD2022-large-trained YOLOv8n
(notebook 09, mAP50-95 0.300 on the RDD2022 test set) and fine-tunes it on
real, manually collected Sofia road photos -- the exact conditions this
project's prototype is meant to work in (`config/app_config.yaml`'s
`pilot_area` is Sofia).

## Data

`data/manual/` (201 verified images) was split (scripts/split_manual_dataset.py,
scripts/split_manual_finetune_val.py) into three non-overlapping parts:

| Split | Images | Role |
|---|---|---|
| fine-tune train | 131 | trains the fine-tuning stage |
| fine-tune val | 28 | monitors fine-tuning for overfitting (early stopping) |
| **held-out test** | **42** | **never trained on by either model** -- used only for the before/after comparison below |

## Why a low learning rate, short run

131 images is tiny relative to the 4,291-image RDD2022-large run. The goal
here isn't to relearn detection from scratch -- it's to nudge the
already-trained weights toward Sofia's specific visual conditions (asphalt
type, lighting, road markings) without destroying what the large RDD2022 run
already learned. A reduced learning rate (0.0005 vs. the from-scratch runs'
0.01) and a short epoch budget with aggressive early stopping (patience=10)
keeps this a genuine fine-tune, not a retrain.

In [1]:
from pathlib import Path

import pandas as pd
from ultralytics import YOLO

DATA_YAML_PATH = Path("../data/manual_finetune_dataset.yaml")
LARGE_MODEL_PATH = Path(
    "runs/detect/runs/road_damage_large/yolov8n_large_first_train/weights/best.pt"
)
DEVICE = 0

## Step 1 -- baseline: evaluate the RDD2022-large model on the held-out Sofia test set (no fine-tuning yet)

This is the "before" half of the comparison.

In [2]:
baseline_model = YOLO(str(LARGE_MODEL_PATH))

baseline_metrics = baseline_model.val(
    data=str(DATA_YAML_PATH),
    split="test",
    imgsz=640,
    batch=8,
    device=DEVICE,
    plots=True,
    project="runs/road_damage_sofia_finetune",
    name="baseline_on_sofia_test",
)

print(f"Baseline (no fine-tuning) on Sofia held-out test set:")
print(f"  mAP50:    {baseline_metrics.box.map50:.4f}")
print(f"  mAP50-95: {baseline_metrics.box.map:.4f}")
print(f"  Precision: {baseline_metrics.box.mp:.3f}")
print(f"  Recall:    {baseline_metrics.box.mr:.3f}")

Ultralytics 8.4.48  Python-3.10.20 torch-2.11.0+cu128 CUDA:0 (NVIDIA GeForce RTX 2070 with Max-Q Design, 8192MiB)
Model summary (fused): 73 layers, 3,006,428 parameters, 0 gradients, 8.1 GFLOPs
val: Fast image access  (ping: 0.20.0 ms, read: 785.8162.2 MB/s, size: 512.9 KB)
val: Scanning C:\Users\vl4di\OneDrive - BUas\GitHub\Vladislav_Stoimenov_Graduation_Project_25_26\data\manual_split\test\labels... 42 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 42/42 205.2it/s 0.2s
val: C:\Users\vl4di\OneDrive - BUas\GitHub\Vladislav_Stoimenov_Graduation_Project_25_26\data\manual_split\test\images\D10_Transverse_Crack_manual_029.jpg: corrupt JPEG restored and saved
val: New cache created: C:\Users\vl4di\OneDrive - BUas\GitHub\Vladislav_Stoimenov_Graduation_Project_25_26\data\manual_split\test\labels.cache
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 6/6 1.1s/it 6.6s
                   all         42        142      0.169      0.2

## Step 2 -- fine-tune on the Sofia fine-tune split

In [3]:
EPOCHS = 30
PATIENCE = 10
IMAGE_SIZE = 640
BATCH_SIZE = 8      # small dataset either way, no resource concern here
LEARNING_RATE = 0.0005  # reduced from the from-scratch runs' 0.01 -- nudge, not retrain

PROJECT_DIR = "runs/road_damage_sofia_finetune"
RUN_NAME = "yolov8n_sofia_finetune"

finetune_model = YOLO(str(LARGE_MODEL_PATH))

finetune_results = finetune_model.train(
    data=str(DATA_YAML_PATH),
    epochs=EPOCHS,
    patience=PATIENCE,
    imgsz=IMAGE_SIZE,
    batch=BATCH_SIZE,
    device=DEVICE,
    workers=0,
    optimizer="SGD",
    lr0=LEARNING_RATE,
    project=PROJECT_DIR,
    name=RUN_NAME,
    exist_ok=True,
)

New https://pypi.org/project/ultralytics/8.4.153 available  Update with 'pip install -U ultralytics'
Ultralytics 8.4.48  Python-3.10.20 torch-2.11.0+cu128 CUDA:0 (NVIDIA GeForce RTX 2070 with Max-Q Design, 8192MiB)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=..\data\manual_finetune_dataset.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=30, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.0005, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=runs\detect\runs\road_damage

## Step 3 -- evaluate the fine-tuned model on the same held-out Sofia test set

In [4]:
RUN_DIR = finetune_model.trainer.save_dir

finetuned_best = YOLO(str(RUN_DIR / "weights" / "best.pt"))

finetuned_metrics = finetuned_best.val(
    data=str(DATA_YAML_PATH),
    split="test",
    imgsz=IMAGE_SIZE,
    batch=BATCH_SIZE,
    device=DEVICE,
    plots=True,
)

print(f"Fine-tuned model on Sofia held-out test set:")
print(f"  mAP50:    {finetuned_metrics.box.map50:.4f}")
print(f"  mAP50-95: {finetuned_metrics.box.map:.4f}")
print(f"  Precision: {finetuned_metrics.box.mp:.3f}")
print(f"  Recall:    {finetuned_metrics.box.mr:.3f}")

Ultralytics 8.4.48  Python-3.10.20 torch-2.11.0+cu128 CUDA:0 (NVIDIA GeForce RTX 2070 with Max-Q Design, 8192MiB)
Model summary (fused): 73 layers, 3,006,428 parameters, 0 gradients, 8.1 GFLOPs
val: Fast image access  (ping: 0.20.0 ms, read: 128.7106.0 MB/s, size: 973.5 KB)
val: Scanning C:\Users\vl4di\OneDrive - BUas\GitHub\Vladislav_Stoimenov_Graduation_Project_25_26\data\manual_split\test\labels.cache... 42 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 42/42  0.0s
val: C:\Users\vl4di\OneDrive - BUas\GitHub\Vladislav_Stoimenov_Graduation_Project_25_26\data\manual_split\test\images\D10_Transverse_Crack_manual_029.jpg: corrupt JPEG restored and saved
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 6/6 1.1s/it 6.4s
                   all         42        142      0.361      0.335      0.278      0.148
D00_Longitudinal_Crack         37         73       0.49      0.699      0.639      0.353
  D10_Transverse_Crack         1

## Before / after comparison -- did domain-adaptation fine-tuning help?

In [5]:
def f1(p, r):
    return 2 * p * r / (p + r + 1e-9)

comparison_df = pd.DataFrame([
    {
        "model": "YOLOv8n RDD2022-large only (no fine-tuning)",
        "precision": round(float(baseline_metrics.box.mp), 3),
        "recall": round(float(baseline_metrics.box.mr), 3),
        "f1": round(f1(baseline_metrics.box.mp, baseline_metrics.box.mr), 3),
        "mAP50": round(float(baseline_metrics.box.map50), 4),
        "mAP50-95": round(float(baseline_metrics.box.map), 4),
    },
    {
        "model": "YOLOv8n RDD2022-large + Sofia fine-tuned",
        "precision": round(float(finetuned_metrics.box.mp), 3),
        "recall": round(float(finetuned_metrics.box.mr), 3),
        "f1": round(f1(finetuned_metrics.box.mp, finetuned_metrics.box.mr), 3),
        "mAP50": round(float(finetuned_metrics.box.map50), 4),
        "mAP50-95": round(float(finetuned_metrics.box.map), 4),
    },
])

map50_95_delta = finetuned_metrics.box.map - baseline_metrics.box.map
print(f"mAP50-95 change from fine-tuning: {map50_95_delta:+.4f} "
      f"({'improved' if map50_95_delta > 0 else 'did not improve'})")

comparison_df

mAP50-95 change from fine-tuning: +0.1072 (improved)


,model,precision,recall,f1,mAP50,mAP50-95
0,YOLOv8n RDD2022-large only (no fine-tuning),0.169,0.215,0.189,0.1105,0.0411
1,YOLOv8n RDD2022-large + Sofia fine-tuned,0.361,0.335,0.347,0.2777,0.1483


## Notes

- Both evaluations use the exact same 42-image held-out Sofia test set --
  neither model has ever seen these images in training, so this is a fair
  before/after comparison.
- If fine-tuning improved results: this directly supports the "the prototype
  should be adapted to its deployment area" argument for the thesis.
- If fine-tuning did *not* improve results (plausible -- 131 images is very
  small, and the model may already generalize reasonably or may overfit to
  the fine-tune set's specific 131 photos): that's still a valid, reportable
  finding, not a failure -- it says something real about how much
  domain-specific data this approach actually needs.
- Same comparison should be run for Faster R-CNN (next notebook) so both
  architectures get evaluated on this axis, not just YOLO.